# MEGA-RAG: Run Evaluation on Colab GPU

Runs the official PubMedQA benchmark on Colab's free T4 GPU using Groq (Llama 3.3 70B).

## What This Does
1. **Clones** the project from GitHub
2. **Installs** dependencies + loads Groq API key
3. **Builds index** — embeds 1000 PubMedQA contexts into ChromaDB + BM25 + Graph (GPU-accelerated)
4. **Evaluates** — runs each config (llm_only, oracle_context, lean_workflow) on test samples
5. **Reports** — accuracy, macro-F1, latency per config

## Prerequisites
- Add `GROQ_API_KEY` as Colab Secret (sidebar key icon 🔑)
- Runtime: **T4 GPU** (Runtime → Change runtime type)

## 1. Clone Project & Setup

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
QUICK_TEST = True          # True = 50 samples (~15 min), False = 500 samples (~2-3 hrs)
SAMPLE_SIZE = 50 if QUICK_TEST else 500
EVAL_CONFIGS = "llm_only,oracle_context,lean_workflow"
ENABLE_CONTEXTUAL = False

# =============================================================================
# CLONE FROM GITHUB
# =============================================================================
import os, sys

PROJECT_DIR = "/content/medicalq-a_rag"
REPO_URL = "https://github.com/Dev07-Harsh/medicalq-a_rag.git"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")
!git log --oneline -3

# GPU check
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_mem', None) or getattr(props, 'total_memory', 0)
    print(f"\nGPU: {torch.cuda.get_device_name(0)} ({vram / 1e9:.1f} GB)")
else:
    print("\nNo GPU — enable in Runtime > Change runtime type > T4 GPU")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print("Dependencies installed!")

In [ ]:
# Removed — deps installed in previous cell
pass

In [ ]:
# =============================================================================
# LOAD GROQ API KEY & CONFIGURE
# =============================================================================
import os

# Create .env on Colab if it doesn't exist (git clone won't include it)
if not os.path.exists('.env'):
    print("No .env file found. Enter your Groq API key:")
    print("(Get one free at https://console.groq.com/keys)")
    api_key = input("GROQ_API_KEY: ").strip()
    with open('.env', 'w') as f:
        f.write(f"GROQ_API_KEY={api_key}\n")
        f.write("LLM_PROVIDER=groq\n")
        f.write("GROQ_MODEL=llama-3.3-70b-versatile\n")
    print(".env file created!")

# Load .env
from dotenv import load_dotenv
load_dotenv('.env')
print(f"API key loaded from .env")

# Groq as primary, optimized for 28 RPM
os.environ['LLM_PROVIDER'] = 'groq'
os.environ['GROQ_MODEL'] = 'llama-3.3-70b-versatile'
os.environ['LLM_AUTO_FALLBACK'] = 'false'
os.environ['ENABLE_CONTEXTUAL_CHUNKING'] = str(ENABLE_CONTEXTUAL).lower()
os.environ['ENABLE_SELF_CONSISTENCY'] = 'false'
os.environ['ENABLE_CITATION_VERIFICATION'] = 'false'
os.environ['ENABLE_BIOMEDICAL_NER'] = 'false'

has_key = bool(os.environ.get('GROQ_API_KEY', ''))
print(f'\nGroq API key: {"SET" if has_key else "MISSING!"}'  )
print(f'Model: llama-3.3-70b-versatile (70B params)')
print(f'Rate limit: 28 RPM (built-in throttle)')

In [ ]:
# Verify data files
import json

print("Data files check:")
for split in ['test', 'dev', 'train_balanced', 'train_oversampled']:
    path = f'pubmedQA/splits/{split}.json'
    if os.path.exists(path):
        with open(path) as f:
            n = len(json.load(f))
        print(f'  [OK] {split}.json: {n} samples')
    else:
        print(f'  [FAIL] {split}.json: NOT FOUND!')

print(f"\nEvaluation plan: {SAMPLE_SIZE} samples, configs: {EVAL_CONFIGS}")

## 2. Build PubMedQA Index

Indexes all 1000 PubMedQA contexts (no answers) into ChromaDB + BM25 + Graph.
Takes ~5-10 min without contextual chunking, ~15-30 min with it.

In [ ]:
%%time
# Build the controlled PubMedQA-only index
# Skip if index already exists (saves time on re-runs)

index_dir = "chroma_pubmedqa_only"
if os.path.exists(index_dir) and os.path.exists(f"{index_dir}/bm25_index.pkl"):
    print(f"Index already exists at {index_dir} — skipping build.")
    print("Delete it and re-run to rebuild: !rm -rf chroma_pubmedqa_only")
else:
    !python build_pubmedqa_index.py --force \
        --persist-dir {index_dir} \
        --collection-name pubmedqa_only
    print('\nIndex built successfully!')

## 3. Run Evaluation

Runs the configured evaluation. Change `QUICK_TEST` and `EVAL_CONFIGS` in cell 1 to control scope.

In [ ]:
%%time
# Run evaluation with configured settings

out_file = f"evaluation_results/eval_{'quick' if QUICK_TEST else 'full'}_{SAMPLE_SIZE}.json"
os.makedirs("evaluation_results", exist_ok=True)

!python evaluate_pubmedqa_multiconfig.py \
    --configs {EVAL_CONFIGS} \
    --test-json pubmedQA/splits/test.json \
    --sample-size {SAMPLE_SIZE} \
    --out {out_file} \
    --seed 42

# Display results
import json
try:
    with open(out_file) as f:
        results = json.load(f)

    print('\n' + '='*72)
    print(f'EVALUATION RESULTS ({SAMPLE_SIZE} samples)')
    print('='*72)
    print(f'{"Config":<20} {"Accuracy":>10} {"Macro-F1":>10} {"Latency":>10} {"Coverage":>10}')
    print('-'*72)
    for name, cfg in results.get('configs', {}).items():
        acc = cfg.get('accuracy', 0)
        f1 = cfg.get('macro_f1', 0)
        lat = cfg.get('avg_latency_s', 0)
        cov = cfg.get('coverage', 0)
        print(f'{name:<20} {acc:>9.1%} {f1:>10.3f} {lat:>9.1f}s {cov:>9.1%}')
    print('='*72)
except FileNotFoundError:
    print(f"Results file not found: {out_file}")

## 4. Full Ablation Study (Optional)

Run ALL configs systematically with LaTeX + CSV output for the paper.
Only run this after the quick test passes.

In [ ]:
%%time
# Full ablation study — all configs, 500 samples
# Uncomment to run (takes 2-4 hours)

# !python scripts/run_ablation_study.py \
#     --test-json pubmedQA/splits/test.json \
#     --sample-size 500 \
#     --out-dir evaluation_results

print("Uncomment the command above to run the full ablation study.")

In [ ]:
# Display ablation results (run after ablation study completes)
import json, os

ablation_path = "evaluation_results/ablation_study.json"
if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        study = json.load(f)

    print('='*72)
    print('ABLATION STUDY RESULTS — Official PubMedQA')
    print('='*72)
    print(f'{"Config":<20} {"Accuracy":>10} {"Macro-F1":>10} {"Latency":>10} {"Correct":>10}')
    print('-'*72)
    for r in sorted(study['results'], key=lambda x: x['accuracy'], reverse=True):
        print(f"{r['config']:<20} {r['accuracy']:>9.1%} {r.get('macro_f1',0):>10.3f} "
              f"{r['avg_latency']:>9.1f}s {r['correct']:>6}/{r['total']}")
    print('='*72)

    # Show LaTeX
    latex_path = "evaluation_results/ablation_latex.tex"
    if os.path.exists(latex_path):
        print('\nLaTeX table:')
        with open(latex_path) as f:
            print(f.read())
else:
    print("No ablation results yet. Run the full ablation study first (cell above).")

## 5. Save Results

In **colab_standalone** mode, copies results to Google Drive.
In **vscode** mode, results are already saved locally in `evaluation_results/`.

In [ ]:
# Download results to your local machine
from google.colab import files
import os

print("Evaluation result files:")
for f in sorted(os.listdir('evaluation_results')):
    if os.path.isfile(f'evaluation_results/{f}'):
        size = os.path.getsize(f'evaluation_results/{f}') / 1024
        print(f'  {f} ({size:.1f} KB)')

# Download the main results file
result_file = f"evaluation_results/eval_{'quick' if QUICK_TEST else 'full'}_{SAMPLE_SIZE}.json"
if os.path.exists(result_file):
    files.download(result_file)
    print(f"\nDownloading: {result_file}")

## 6. (Optional) Re-run with Contextual Chunking

Rebuild the index WITH contextual chunking, then evaluate again.
This uses ~1000 LLM API calls for indexing — make sure you have quota.

In [ ]:
# %%time
# # Uncomment this entire cell to run with contextual chunking
# # WARNING: Uses ~1000 LLM API calls for context generation
#
# import os
# os.environ['ENABLE_CONTEXTUAL_CHUNKING'] = 'true'
#
# # Rebuild index with contextual descriptions
# !python build_pubmedqa_index.py --force \
#     --persist-dir chroma_pubmedqa_contextual \
#     --collection-name pubmedqa_contextual
#
# # Run evaluation on contextual index
# !python evaluate_pubmedqa_multiconfig.py \
#     --configs oracle_context,hybrid,lean_workflow \
#     --test-json pubmedQA/splits/test.json \
#     --sample-size {SAMPLE_SIZE} \
#     --persist-dir chroma_pubmedqa_contextual \
#     --collection-name pubmedqa_contextual \
#     --out evaluation_results/eval_contextual.json
#
# # Compare results
# import json
# for name, path in [("Without contextual", out_file), ("With contextual", "evaluation_results/eval_contextual.json")]:
#     if os.path.exists(path):
#         with open(path) as f:
#             data = json.load(f)
#         print(f"\n{name}:")
#         for cfg_name, cfg in data.get("configs", {}).items():
#             print(f"  {cfg_name}: acc={cfg.get('accuracy',0):.1%} F1={cfg.get('macro_f1',0):.3f}")

print("Uncomment cell above to test contextual chunking improvement.")